In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
from smart_grid_lab.core import Time

# Load data

In [3]:
pv_df = pd.read_csv("data/pv_kW.csv")

In [4]:
pv_df.index = pd.to_datetime(pv_df.utc_timestamp, utc=True)
pv_df = pv_df.drop("utc_timestamp", axis='columns')

In [5]:
TARGET = "pv"

# Prepare Dataset

In [6]:
import smart_grid_lab.data_pipeline.features as features

In [7]:
pv_df = features.append_time_features(pv_df, TARGET)

In [8]:
pv_df.head()

,pv,hour,dayofweek,month,dayofyear,weekofyear,is_weekend,hour_sin,hour_cos,dow_sin,dow_cos,load_lag_1h,load_lag_6h,load_lag_24h,load_lag_48h,load_lag_7d,load_roll_mean_24h,load_roll_std_24h
utc_timestamp,,,,,,,,,,,,,,,,,,
2014-12-18 18:00:00+00:00,0.0,18,3,12,352,51,0,-1.000000,-1.836970e-16,0.433884,-0.900969,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2014-12-18 18:15:00+00:00,0.0,18,3,12,352,51,0,-1.000000,-1.836970e-16,0.433884,-0.900969,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2014-12-18 18:30:00+00:00,0.0,18,3,12,352,51,0,-1.000000,-1.836970e-16,0.433884,-0.900969,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2014-12-18 18:45:00+00:00,0.0,18,3,12,352,51,0,-1.000000,-1.836970e-16,0.433884,-0.900969,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2014-12-18 19:00:00+00:00,0.0,19,3,12,352,51,0,-0.965926,2.588190e-01,0.433884,-0.900969,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [9]:
openstef_pv_df = pv_df.reset_index()

openstef_pv_df.rename(
    columns={
        pv_df.index.name: "datetime",
        TARGET: TARGET,
    },
    inplace=True,
)

openstef_pv_df = openstef_pv_df.set_index("datetime")

openstef_pv_df.head()

,pv,hour,dayofweek,month,dayofyear,weekofyear,is_weekend,hour_sin,hour_cos,dow_sin,dow_cos,load_lag_1h,load_lag_6h,load_lag_24h,load_lag_48h,load_lag_7d,load_roll_mean_24h,load_roll_std_24h
datetime,,,,,,,,,,,,,,,,,,
2014-12-18 18:00:00+00:00,0.0,18,3,12,352,51,0,-1.000000,-1.836970e-16,0.433884,-0.900969,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2014-12-18 18:15:00+00:00,0.0,18,3,12,352,51,0,-1.000000,-1.836970e-16,0.433884,-0.900969,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2014-12-18 18:30:00+00:00,0.0,18,3,12,352,51,0,-1.000000,-1.836970e-16,0.433884,-0.900969,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2014-12-18 18:45:00+00:00,0.0,18,3,12,352,51,0,-1.000000,-1.836970e-16,0.433884,-0.900969,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2014-12-18 19:00:00+00:00,0.0,19,3,12,352,51,0,-0.965926,2.588190e-01,0.433884,-0.900969,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [10]:
from openstef_core.datasets import TimeSeriesDataset

In [11]:
from datetime import datetime, timedelta

In [12]:
dataset = TimeSeriesDataset(openstef_pv_df, sample_interval=timedelta(minutes=15))

In [13]:
dataset.feature_names

['pv',
 'hour',
 'dayofweek',
 'month',
 'dayofyear',
 'weekofyear',
 'is_weekend',
 'hour_sin',
 'hour_cos',
 'dow_sin',
 'dow_cos',
 'load_lag_1h',
 'load_lag_6h',
 'load_lag_24h',
 'load_lag_48h',
 'load_lag_7d',
 'load_roll_mean_24h',
 'load_roll_std_24h']

In [14]:
train_start = datetime.fromisoformat("2016-03-15T00:00:00Z")
train_end = train_start + timedelta(days=45)
forecast_end = train_end + timedelta(days=7)

In [15]:
train_dataset = dataset.filter_by_range(start=train_start, end=train_end)

In [16]:
# Include 14 days of history before forecast start for lag feature computation
predict_dataset = dataset.filter_by_range(
    start=train_end - timedelta(days=14),
    end=forecast_end,
)

In [17]:
print(
    f"Training:  {train_dataset.data.shape[0]:,} rows, "
    f"{train_dataset.data.index.min():%Y-%m-%d} to {train_dataset.data.index.max():%Y-%m-%d}"
)

Training:  4,320 rows, 2016-03-15 to 2016-04-28


In [18]:
print(
    f"Predict:   {predict_dataset.data.shape[0]:,} rows, "
    f"{predict_dataset.data.index.min():%Y-%m-%d} to {predict_dataset.data.index.max():%Y-%m-%d}"
)

Predict:   2,016 rows, 2016-04-15 to 2016-05-05


In [26]:
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "browser"

In [32]:
fig = px.line(
    train_dataset.data,
    x=train_dataset.data.index,
    y="pv",
    title="Training interval — pv",
    labels={
        "x": "Time",
        "pv": "PV (MW)",
    },
)

In [33]:
fig = fig.update_layout(
    yaxis_title="PV (MW)",
    xaxis_title="Time",
)

In [34]:
fig.show()

# Configure Storage (MLFlow) — fixes my_mlflow_storage


In [ ]:
# MLFlowStorage init — same instance will be reused in infrastructure/inference
# openstef_models 4.x uses file store; allow it explicitly (mlflow >2.9 maintenance mode)
import os
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"

from pathlib import Path
from openstef_models.integrations.mlflow.mlflow_storage import MLFlowStorage
from openstef_models.integrations.mlflow.mlflow_storage_callback import MLFlowStorageCallback

my_mlflow_storage = MLFlowStorage(
    tracking_uri="./mlflow",  # -> file:///abs/path/mlflow via normalize_tracking_uri
    local_artifacts_path=Path("./mlflow_artifacts_local"),
    # For production DB backend use:
    # tracking_uri="sqlite:///mlflow.db",
    # artifact_location="file:///tmp/mlflow_artifacts",
)
callback = MLFlowStorageCallback(
    storage=my_mlflow_storage,
    model_reuse_enable=True,
    model_reuse_max_age=timedelta(days=7),
)
print("MLFlow storage:", my_mlflow_storage.tracking_uri)
print("callback storage is same:", callback.storage is my_mlflow_storage)


# Weather: versioned NWP (Liander reference + OpenWeather)
OpenSTEF expects weather as columns inside a VersionedTimeSeriesDataset with `available_at`.
We load the Liander reference parquet (timestamp, available_at, shortwave_radiation, temperature_2m, ...)
and compose it with PV measurements. For operational use replace with `OpenWeatherAdapter`.


In [ ]:
from pathlib import Path

# Option A: Liander versioned weather (benchmark reference, already in repo)
try:
    from smart_grid_lab.infrastructure.weather.liander_weather_adapter import load_liander_weather_dataset
    weather_ds = load_liander_weather_dataset(
        "liander_dataset/weather_forecasts_versioned/mv_feeder/OS Gorredijk.parquet"
    )
    print("Weather (Liander) loaded:", weather_ds.data.shape, weather_ds.feature_names[:6])
    print(weather_ds.data.head(2).to_string())
except Exception as e:
    print("Liander weather not found, falling back to synthetic:", e)
    # Fallback: create synthetic weather aligned to pv index for demo (15min)
    import numpy as np
    idx = openstef_pv_df.index
    # simple synthetic GHI: clearsky-like sinusoid + noise
    hour = idx.hour + idx.minute/60
    synthetic_ghi = (800 * np.sin(np.pi * (hour-6)/12).clip(min=0) + np.random.normal(0, 30, len(idx))).clip(min=0)
    wdf = pd.DataFrame({
        "shortwave_radiation": synthetic_ghi,
        "temperature_2m": 15 + 5*np.sin(2*np.pi*idx.dayofyear/365) + np.random.normal(0,1,len(idx)),
        "relative_humidity_2m": 70 + np.random.normal(0,5,len(idx)),
        "surface_pressure": 1013 + np.random.normal(0,3,len(idx)),
        "wind_speed_10m": 5 + np.random.normal(0,1,len(idx)),
        "cloud_cover": 50 + np.random.normal(0,10,len(idx)),
        "available_at": idx,  # naive: available at same time (no forecast lag for demo)
    }, index=idx)
    from openstef_core.datasets import TimeSeriesDataset as TSD
    weather_ds = TSD(wdf, sample_interval=timedelta(minutes=15), available_at_column="available_at")
    print("Synthetic weather:", weather_ds.data.shape)

# Option B: live OpenWeather (requires OPENWEATHER_API_KEY) — uncomment to use
# from smart_grid_lab.infrastructure.weather.openweather_adapter import OpenWeatherAdapter
# ow = OpenWeatherAdapter(lat=52.37, lon=4.90)  # Amsterdam
# weather_ds = ow.fetch_versioned_dataset()  # adds available_at = now


In [ ]:
# Compose PV + weather into VersionedTimeSeriesDataset (disjoint columns, same 15min interval)
from openstef_core.datasets import VersionedTimeSeriesDataset

# pv part needs available_at as well for versioned composition — wrap pv as versioned with trivial available_at
from openstef_core.datasets import TimeSeriesDataset as TSD

# If pv dataset already versioned? check
print("pv dataset is_versioned:", dataset.is_versioned)
print("weather is_versioned:", weather_ds.is_versioned)

# Create versioned wrappers: pv measurements are "actuals" -> available_at = timestamp
pv_versioned = TSD(
    dataset.data.assign(available_at=dataset.data.index),  # available at observation time
    sample_interval=timedelta(minutes=15),
    available_at_column="available_at",
)
weather_versioned = weather_ds  # already versioned

versioned_dataset = VersionedTimeSeriesDataset(data_parts=[pv_versioned, weather_versioned])
print("Versioned dataset parts:", len(versioned_dataset.data_parts))
print("Features:", versioned_dataset.feature_names[:12])
print("Index range:", versioned_dataset.index.min(), "->", versioned_dataset.index.max())

# For training we restrict to pv period and Liander weather starts 2024 — so for synthetic pv (2014-2018)
# the Liander weather won't overlap. In that case fall back to non-versioned merged DataFrame for demo:
# VersionedTimeSeriesDataset has no .data; use index/feature check
needs_fallback = weather_ds.index.min() > train_end  # Liander 2024 vs pv 2016 -> no overlap -> use synthetic
if needs_fallback:
    print("Note: Liander weather (2024) does not overlap pv (2014-2018) — using fallback merged non-versioned dataset for training")
    # merge on index inner join for demo
    try:
        merged = openstef_pv_df.join(weather_ds.data.drop(columns=["available_at"], errors="ignore"), how="inner")
    except Exception:
        merged = openstef_pv_df.copy()
    if merged.empty:
        # use synthetic or just pv_df alone (workflow will log warning for missing weather but still train)
        merged = openstef_pv_df.copy()
        # add dummy weather columns so pipeline doesn't warn
        for col in ["shortwave_radiation","temperature_2m","relative_humidity_2m","surface_pressure","wind_speed_10m"]:
            if col not in merged.columns:
                # synthesize plausible values for demo
                import numpy as np
                if col == "shortwave_radiation":
                    hour = merged.index.hour + merged.index.minute/60
                    merged[col] = np.clip(800 * np.sin(np.pi * (hour-6)/12), 0, None)
                elif col == "temperature_2m":
                    merged[col] = 15.0
                elif col == "relative_humidity_2m":
                    merged[col] = 70.0
                elif col == "surface_pressure":
                    merged[col] = 1013.0
                elif col == "wind_speed_10m":
                    merged[col] = 5.0
    from openstef_core.datasets import TimeSeriesDataset as TSD2
    train_versioned_dataset = TSD2(merged, sample_interval=timedelta(minutes=15))
    # rebuild train/predict splits from this
    train_dataset = train_versioned_dataset.filter_by_range(start=train_start, end=train_end)
    predict_dataset = train_versioned_dataset.filter_by_range(start=train_end - timedelta(days=14), end=forecast_end)
else:
    train_versioned_dataset = versioned_dataset
    train_dataset = train_versioned_dataset.filter_by_range(start=train_start, end=train_end)
    predict_dataset = train_versioned_dataset.filter_by_range(start=train_end - timedelta(days=14), end=forecast_end)

print(f"Training: {train_dataset.data.shape[0]} rows")
print(f"Predict: {predict_dataset.data.shape[0]} rows")
print("Columns:", train_dataset.feature_names[:10])


# Workflow config — weather-aware GBLinear + MLFlow reuse


In [ ]:
from openstef_core.types import LeadTime, Q
from openstef_models.presets import ForecastingWorkflowConfig, create_forecasting_workflow
from openstef_models.presets.forecasting_workflow import GBLinearForecaster
from pydantic_extra_types.coordinate import Coordinate, Latitude, Longitude
from decimal import Decimal
from openstef_models.presets.forecasting_workflow import LocationConfig

# Two modes: direct PV forecast vs two-stage (forecast GHI then physics). Toggle:
TWO_STAGE = False  # set True to train on radiation and apply linear pv conversion

config = ForecastingWorkflowConfig(
    model_id="pv_gblinear_ghi" if TWO_STAGE else "pv_gblinear",
    model="gblinear",
    horizons=[LeadTime.from_string("PT36H")],
    quantiles=[Q(0.1), Q(0.5), Q(0.9)],
    target_column="shortwave_radiation" if TWO_STAGE else "pv",
    # Liander / OpenWeather column names -> must match dataset
    temperature_column="temperature_2m",
    relative_humidity_column="relative_humidity_2m",
    wind_speed_column="wind_speed_10m",
    radiation_column="shortwave_radiation",
    pressure_column="surface_pressure",
    location=LocationConfig(coordinate=Coordinate(latitude=Latitude(Decimal("52.132633")), longitude=Longitude(Decimal("5.291266")))),
    verbosity=0,
    mlflow_storage=my_mlflow_storage,  # preferred over manual callbacks list since 4.2
    # manual callback also works: callbacks=[callback] via ForecastingWorkflowConfig.callbacks auto-created
    gblinear_hyperparams=GBLinearForecaster.HyperParams(n_steps=50),
)
print(config.model_id, config.target_column, config.radiation_column)


In [ ]:
workflow = create_forecasting_workflow(config)
print("Workflow model_id:", workflow.model_id)
print("Callbacks:", workflow.callbacks)


# Train


In [ ]:
result = workflow.fit(train_dataset)
if result is not None:
    print("Training metrics:")
    print(result.metrics_full.to_dataframe())
    if result.metrics_test is not None:
        print("\nTest metrics:")
        print(result.metrics_test.to_dataframe())
else:
    print("Fit skipped (model reuse, recent enough)")


# Predict (workflow handles MLFlow load if not fitted)


In [ ]:
from openstef_core.datasets import ForecastDataset

forecast: ForecastDataset = workflow.predict(predict_dataset, forecast_start=train_end)
print(f"Forecast rows: {len(forecast.data)}, quantiles: {forecast.quantiles}")
forecast.data.tail()


# Two-stage: irradiance -> PV physics (linear det. conversion)
Keep irradiance forecast separate from PV conversion so capacity/tilt changes don't require retraining.
Direct PV forecast learns per-site effects; two-stage is site-agnostic and recommended for smart-grid-lab.


In [ ]:
from smart_grid_lab.infrastructure.forecasting.pv_physics import PVSystemConfig, ghi_to_pv_linear, apply_pv_physics_to_forecast

pv_config = PVSystemConfig(kWp=10.0, use_pvlib=False, temp_coeff=-0.004)

# If TWO_STAGE, forecast is GHI -> convert
if TWO_STAGE:
    pv_forecast_df = apply_pv_physics_to_forecast(forecast.data, pv_config, ghi_column="shortwave_radiation")
    print("PV from GHI (two-stage) tail:")
    print(pv_forecast_df.tail().to_string())
else:
    # Demo: even for direct PV, show linear conversion of NWP GHI as baseline
    # Use weather shortwave_radiation at forecast horizon as naive PV
    sample_ghi = predict_dataset.data["shortwave_radiation"].tail(5) if "shortwave_radiation" in predict_dataset.data.columns else forecast.data["quantile_P50"].tail(5)
    pv_linear = ghi_to_pv_linear(sample_ghi, kWp=pv_config.kWp)
    print("Linear PV from GHI (baseline):")
    print(pv_linear.to_string())
    print("\nForecast PV (ML, direct):")
    print(forecast.data["quantile_P50"].tail().to_string())

# Example: scale with different capacity without retraining
pv_config_20kWp = PVSystemConfig(kWp=20.0)
print("\nSame GHI -> 20kWp:", ghi_to_pv_linear(800.0, kWp=20.0), "kW")


# Reuse in infrastructure/inference — same MLFlowStorage


In [ ]:
# This is how microgrid simulation / controller will load the model without retraining
from smart_grid_lab.infrastructure.forecasting.openstef_pv_adapter import OpenSTEFGBLinearPVForecastAdapter
from smart_grid_lab.infrastructure.forecasting.pv_physics import PVSystemConfig as PVConf

# Reuse same storage + model_id (no new training)
pv_adapter = OpenSTEFGBLinearPVForecastAdapter(
    mlflow_storage=my_mlflow_storage,
    model_id="pv_gblinear_ghi" if TWO_STAGE else "pv_gblinear",
    pv_config=PVConf(kWp=10.0),
    two_stage=TWO_STAGE,
)

# Predict via adapter (internally recreates workflow, MLFlowStorageCallback loads latest run)
adapter_forecast = pv_adapter.predict_quantiles(predict_dataset, forecast_start=train_end)
print("Adapter forecast rows:", adapter_forecast.data.shape)
print(adapter_forecast.data.tail().to_string())

# Point forecast for controller
at = train_end
pv_kw = pv_adapter.forecast_pv_kw(predict_dataset, at=pd.Timestamp(at))
print(f"\nPV at {at}: {pv_kw:.3f} kW (P50)")

# Direct storage access also works
runs = my_mlflow_storage.search_latest_runs("pv_gblinear_ghi" if TWO_STAGE else "pv_gblinear")
print("Latest MLflow run:", runs[0].info.run_id if runs else "none")


# Visualize (browser renderer)


In [ ]:
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "browser"

fig = px.line(train_dataset.data, x=train_dataset.data.index, y="pv" if "pv" in train_dataset.data.columns else "shortwave_radiation", title="Training interval — target")
fig.update_layout(yaxis_title="PV (kW) / GHI (W/m2)", xaxis_title="Time")
fig.show()

# Forecast vs actual tail
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(10,3))
    plt.plot(forecast.data.index, forecast.data["quantile_P50"], label="P50 forecast")
    plt.fill_between(forecast.data.index, forecast.data["quantile_P10"], forecast.data["quantile_P90"], alpha=0.2, label="P10-P90")
    if "pv" in predict_dataset.data.columns:
        plt.plot(predict_dataset.data.index, predict_dataset.data["pv"], alpha=0.6, label="actual pv")
    plt.legend(); plt.title("Forecast vs actual (tail)"); plt.show()
except Exception as e:
    print("plot fallback:", e)
    print(forecast.data.tail())
